In [ ]:
!pip install edge-tts==7.0.2 ffmpeg-python

In [ ]:
#!/usr/bin/env python3
"""
Local TTS Script

Converts text files to audio using edge-tts.
Simplified for local use without Google Drive/Colab dependencies.

Usage:
    python tts_local.py input.txt output_dir/
    python tts_local.py input_dir/ output_dir/
    python tts_local.py --retry output_dir/

Requirements:
    pip install edge-tts ffmpeg-python
"""

import os
import asyncio
import shutil
import argparse
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional
import edge_tts
import logging
import json
import hashlib
from datetime import datetime

In [ ]:
logger = logging.getLogger(__name__)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class TTSConfig:
    """TTS processing parameters."""
    voice: str = "en-US-SteffanNeural"
    speed_multiplier: float = 1
    max_concurrent: int = 5
    retries: int = 3
    chunk_size: int = 500
    retry_base_delay: float = 2.0

    def __post_init__(self):
        if not 0.5 <= self.speed_multiplier <= 2.0:
            raise ValueError("speed_multiplier should be between 0.5 and 2.0")
        if self.max_concurrent < 1:
            raise ValueError("max_concurrent must be at least 1")
        if self.retries < 1:
            raise ValueError("retries must be at least 1")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Utilities
# ─────────────────────────────────────────────────────────────────────────────
def setup_logging(log_file: Optional[Path] = None, level: int = logging.INFO) -> None:
    """Configure logging output."""
    handlers = []
    if log_file:
        handlers.append(logging.FileHandler(str(log_file)))
    else:
        handlers.append(logging.StreamHandler())

    logging.basicConfig(
        level=level,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=handlers,
        force=True
    )

In [ ]:
def get_file_hash(file_path: Path) -> str:
    """Compute MD5 hash of a file."""
    with open(file_path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

In [ ]:



def setup_output_structure(output_base: Path) -> tuple[Path, Path, Path]:
    """Create output directory structure: Full_Audio, Single_Files, Zips."""
    full_audio_dir = output_base / "Full_Audio"
    single_files_dir = output_base / "Single_Files"
    zips_dir = single_files_dir / "Zips"
    for d in [full_audio_dir, single_files_dir, zips_dir]:
        d.mkdir(parents=True, exist_ok=True)
    return full_audio_dir, single_files_dir, zips_dir

In [ ]:

def create_run_log(full_audio_dir: Path, run_info: dict) -> Path:
    """Save run metadata to a timestamped JSON log."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = full_audio_dir / f"run_{timestamp}.json"
    with open(log_file, 'w') as f:
        json.dump(run_info, f, indent=2)
    return log_file


In [ ]:

def chunk_text(text: str, chunk_size: int = 500) -> list[str]:
    """Split text into word-based chunks."""
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

In [ ]:


# ─────────────────────────────────────────────────────────────────────────────
# Audio Processing
# ─────────────────────────────────────────────────────────────────────────────
async def async_save(
    text_chunk: str,
    voice: str,
    rate: str,
    file_path: Path,
    index: int,
    total: int,
    semaphore: asyncio.Semaphore,
    config: TTSConfig
) -> None:
    """Convert a single text chunk to audio with retry logic."""
    async with semaphore:
        print(f"Starting chunk {index + 1}/{total} -> {file_path.name}")
        communicate = edge_tts.Communicate(text_chunk, voice, rate=rate)
        for attempt in range(config.retries):
            try:
                await communicate.save(str(file_path))
                print(f"Finished chunk {index + 1}/{total} -> {file_path.name}")
                return
            except Exception as e:
                logger.error(f"Chunk {index + 1} failed attempt {attempt + 1}: {type(e).__name__}: {str(e)}")
                print(f"--- WARNING: Chunk {index + 1} failed attempt {attempt + 1} ---")
                if attempt < config.retries - 1:
                    delay = config.retry_base_delay * (2 ** attempt)
                    await asyncio.sleep(delay)
                else:
                    raise e

In [ ]:
async def process_text_to_audio_chunks(
    text_chunks: list[str],
    input_file: Path,
    config: TTSConfig,
    output_dir: Path
) -> tuple[list[int], Path, Path]:
    """Process all text chunks to audio files concurrently."""
    output_dir.mkdir(parents=True, exist_ok=True)
    log_file = output_dir / "processing.log"
    setup_logging(log_file)

    rate_percentage = int((config.speed_multiplier - 1.0) * 100)
    rate_string = f"+{rate_percentage}%" if rate_percentage >= 0 else f"{rate_percentage}%"

    semaphore = asyncio.Semaphore(config.max_concurrent)
    tasks = []
    for i, chunk in enumerate(text_chunks):
        file_path = output_dir / f"{i}.flac"
        task = async_save(chunk, config.voice, rate_string, file_path, i, len(text_chunks), semaphore, config)
        tasks.append(task)

    results = await asyncio.gather(*tasks, return_exceptions=True)
    failed_chunks = [i for i, r in enumerate(results) if isinstance(r, Exception)]
    for i, r in enumerate(results):
        if isinstance(r, Exception):
            logger.error(f"Final result for chunk {i+1}: {type(r).__name__}: {str(r)}")

    failed_json = output_dir / "failed_chunks.json"
    with open(failed_json, 'w') as f:
        json.dump({
            "input_file": str(input_file),
            "input_hash": get_file_hash(input_file),
            "failed_chunks": failed_chunks
        }, f)

    return failed_chunks, log_file, failed_json

In [ ]:
def concatenate_audio_files(input_dir: Path, output_file: Path) -> bool:
    """Concatenate FLAC chunks into a single audio file using ffmpeg."""
    try:
        import ffmpeg
    except ImportError:
        raise ImportError("ffmpeg-python not installed. Run: pip install ffmpeg-python")

    files = sorted(input_dir.glob("*.flac"), key=lambda f: int(f.stem))
    if not files:
        print("No audio files to concatenate.")
        return False

    try:
        inputs = [ffmpeg.input(str(f)) for f in files]
        ffmpeg.concat(*inputs, v=0, a=1).output(str(output_file)).run(quiet=True, overwrite_output=True)
        print(f"Concatenated to {output_file}")
        return True
    except ffmpeg.Error as e:
        logger.error(f"FFmpeg error during concatenation: {e.stderr.decode() if e.stderr else str(e)}")
        print(f"ERROR: Failed to concatenate audio files: {e}")
        return False

In [ ]:
def create_zip(source_dir: Path, zip_path: Path) -> bool:
    """Create a zip archive of audio chunks."""
    try:
        zip_base = zip_path.with_suffix('')
        shutil.make_archive(str(zip_base), 'zip', source_dir)
        print(f"Created zip: {zip_path}")
        return True
    except Exception as e:
        logger.error(f"Failed to create zip: {e}")
        print(f"ERROR: Failed to create zip archive: {e}")
        return False

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Main Processing Functions
# ─────────────────────────────────────────────────────────────────────────────
async def process_single_text_file(
    input_file: Path,
    output_base: Path,
    config: Optional[TTSConfig] = None
) -> dict:
    """Process a single text file to audio."""
    if config is None:
        config = TTSConfig()

    full_audio_dir, single_files_dir, zips_dir = setup_output_structure(output_base)
    stem = input_file.stem
    single_dir = single_files_dir / stem
    single_dir.mkdir(exist_ok=True)

    with open(input_file, 'r', encoding='utf-8') as f:
        text = f.read()

    text_chunks = chunk_text(text, config.chunk_size)
    failed_chunks, log_file, failed_json = await process_text_to_audio_chunks(
        text_chunks, input_file, config, single_dir
    )

    success = len(failed_chunks) == 0
    full_file = None
    zip_file = None

    if success:
        full_file = full_audio_dir / f"{stem}.flac"
        concatenate_audio_files(single_dir, full_file)
        zip_file = zips_dir / f"{stem}.zip"
        create_zip(single_dir, zip_file)

    return {
        "input_file": str(input_file),
        "input_hash": get_file_hash(input_file),
        "output_base": str(output_base),
        "success": success,
        "failed_chunks": failed_chunks,
        "log_file": str(log_file),
        "failed_json": str(failed_json),
        "full_file": str(full_file) if full_file else None,
        "zip_file": str(zip_file) if zip_file else None
    }

In [ ]:


async def process_directory(
    input_dir: Path,
    output_base: Path,
    config: Optional[TTSConfig] = None
) -> tuple[dict, Path]:
    """Process all .txt files in a directory."""
    if config is None:
        config = TTSConfig()

    full_audio_dir, single_files_dir, zips_dir = setup_output_structure(output_base)
    txt_files = list(input_dir.glob("*.txt"))

    run_info = {
        "run_timestamp": datetime.now().isoformat(),
        "input_dir": str(input_dir),
        "output_base": str(output_base),
        "config": asdict(config),
        "files_processed": []
    }

    for txt_file in txt_files:
        result = await process_single_text_file(txt_file, output_base, config)
        run_info["files_processed"].append(result)

    run_log = create_run_log(full_audio_dir, run_info)
    return run_info, run_log

In [ ]:

async def retry_failed_chunks_from_json(
    failed_json: Path,
    config: Optional[TTSConfig] = None
) -> list[int]:
    """Retry processing failed chunks from a failed_chunks.json file."""
    if config is None:
        config = TTSConfig()

    with open(failed_json, 'r') as f:
        data = json.load(f)

    input_file = Path(data["input_file"])
    failed = data["failed_chunks"]

    if not failed:
        print("No failed chunks to retry.")
        return []

    with open(input_file, 'r', encoding='utf-8') as f:
        text = f.read()

    text_chunks = chunk_text(text, config.chunk_size)
    output_dir = failed_json.parent

    rate_percentage = int((config.speed_multiplier - 1.0) * 100)
    rate_string = f"+{rate_percentage}%" if rate_percentage >= 0 else f"{rate_percentage}%"

    semaphore = asyncio.Semaphore(config.max_concurrent)
    tasks = []
    for i in failed:
        if i >= len(text_chunks):
            print(f"Invalid chunk index {i}")
            continue
        chunk = text_chunks[i]
        file_path = output_dir / f"{i}.flac"
        task = async_save(chunk, config.voice, rate_string, file_path, i, len(text_chunks), semaphore, config)
        tasks.append(task)

    results = await asyncio.gather(*tasks, return_exceptions=True)
    new_failed = [failed[i] for i, r in enumerate(results) if isinstance(r, Exception)]

    with open(failed_json, 'w') as f:
        json.dump({
            "input_file": str(input_file),
            "input_hash": data["input_hash"],
            "failed_chunks": new_failed
        }, f)

    if not new_failed:
        full_audio_dir = output_dir.parent.parent / "Full_Audio"
        full_file = full_audio_dir / f"{input_file.stem}.flac"
        concatenate_audio_files(output_dir, full_file)
        zips_dir = output_dir.parent / "Zips"
        zip_file = zips_dir / f"{input_file.stem}.zip"
        create_zip(output_dir, zip_file)
        print("All chunks succeeded, created full audio and zip.")

    return new_failed

In [ ]:
def retry_failures(output_base: Path, config: Optional[TTSConfig] = None) -> None:
    """Find and retry all failed chunks in an output directory."""
    if config is None:
        config = TTSConfig()

    single_files_dir = output_base / "Single_Files"
    failed_jsons = list(single_files_dir.rglob("failed_chunks.json"))

    if not failed_jsons:
        print("No failed chunks found!")
        return

    print(f"Found {len(failed_jsons)} file(s) with failures. Retrying...")

    for failed_json in failed_jsons:
        print(f"\nRetrying: {failed_json.parent.name}")
        remaining = asyncio.run(retry_failed_chunks_from_json(failed_json, config))
        if remaining:
            print(f"  Still have {len(remaining)} failed chunks")
        else:
            print(f"  ✓ All chunks succeeded!")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Interactive Helpers (for use from Python REPL or scripts)
# ─────────────────────────────────────────────────────────────────────────────
async def process_file(
    input_file: str,
    output_dir: str,
    voice: str = "en-US-SteffanNeural",
    speed: float = 1,
    concurrent: int = 5,
    chunk_size: int = 500
) -> dict:
    """
    Process a single text file to audio.

    Args:
        input_file: Path to input .txt file
        output_dir: Output directory path
        voice: TTS voice ID (default: en-US-SteffanNeural)
        speed: Speed multiplier 0.5-2.0 (default: 1)
        concurrent: Max concurrent API requests (default: 5)
        chunk_size: Words per chunk (default: 500)

    Returns:
        Dict with processing results

    Example (Jupyter/async context):
        >>> result = await process_file("book.txt", "./output")
        >>> print(result["full_file"])

    Example (regular Python script):
        >>> import asyncio
        >>> result = asyncio.run(process_file("book.txt", "./output"))
    """
    config = TTSConfig(
        voice=voice,
        speed_multiplier=speed,
        max_concurrent=concurrent,
        chunk_size=chunk_size
    )
    return await process_single_text_file(Path(input_file), Path(output_dir), config)


In [ ]:
def process_dir(
    input_dir: str,
    output_dir: str,
    voice: str = "en-US-SteffanNeural",
    speed: float = 1,
    concurrent: int = 5,
    chunk_size: int = 500
) -> tuple[dict, Path]:
    """
    Process all .txt files in a directory.

    Args:
        input_dir: Directory containing .txt files
        output_dir: Output directory path
        voice: TTS voice ID (default: en-US-SteffanNeural)
        speed: Speed multiplier 0.5-2.0 (default: 1)
        concurrent: Max concurrent API requests (default: 5)
        chunk_size: Words per chunk (default: 500)

    Returns:
        Tuple of (run_info dict, run_log path)

    Example:
        >>> from tts_local import process_dir
        >>> info, log = process_dir("./texts", "./output")
    """
    config = TTSConfig(
        voice=voice,
        speed_multiplier=speed,
        max_concurrent=concurrent,
        chunk_size=chunk_size
    )
    return asyncio.run(process_directory(Path(input_dir), Path(output_dir), config))

In [ ]:
def retry(output_dir: str, voice: str = "en-US-SteffanNeural", speed: float = 1) -> None:
    """
    Retry all failed chunks in an output directory.

    Args:
        output_dir: Output directory containing Single_Files/
        voice: TTS voice ID (default: en-US-SteffanNeural)
        speed: Speed multiplier 0.5-2.0 (default: 1)

    Example:
        >>> from tts_local import retry
        >>> retry("./output")
    """
    config = TTSConfig(voice=voice, speed_multiplier=speed)
    retry_failures(Path(output_dir), config)

In [ ]:
# Single file
await process_file("Holo.txt", "./output")

In [ ]:
# All .txt files in a directory
info, log = await process_dir("./texts", "./output")

In [ ]:
# Retry failed chunks
await retry("./output")